In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark", "datasets", "wordcloud", "mlflow"])
    from google.colab import drive
    drive.mount("/content/drive")

import os
if IN_COLAB:
    os.chdir("/content/drive/MyDrive/amazon-reviews-sentiment-analysis")
else:
    # Ensure src/ is importable
    sys.path.insert(0, os.path.abspath("..")) if os.path.basename(os.getcwd()) == "notebooks" else None

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer as SklearnCV

# Exploratory Data Analysis - Amazon Reviews Polarity

In [ ]:
from src.config import DataConfig
from src.data_loader import load_data_pandas

config = DataConfig()
train_df, test_df = load_data_pandas(config, sample_size=500_000)
print(f"Train: {len(train_df):,} rows | Test: {len(test_df):,} rows")
train_df.head()

## Class Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, df) in zip(axes, [("Train", train_df), ("Test", test_df)]):
    counts = df["label"].value_counts().sort_index()
    ax.bar(["Negative", "Positive"], counts.values, color=["#e74c3c", "#2ecc71"])
    ax.set_title(f"{name} Set Class Distribution")
    ax.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 1000, f"{v:,}", ha="center")
plt.tight_layout()
plt.savefig("docs/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Review Length Distribution

In [ ]:
train_df["char_length"] = train_df["combined_text"].str.len()
train_df["word_count"] = train_df["combined_text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train_df["char_length"].clip(upper=3000), bins=100, color="steelblue", edgecolor="none", alpha=0.8)
axes[0].set_title("Character Length Distribution")
axes[0].set_xlabel("Character Length")
axes[0].set_ylabel("Count")

axes[1].hist(train_df["word_count"].clip(upper=500), bins=100, color="coral", edgecolor="none", alpha=0.8)
axes[1].set_title("Word Count Distribution")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.savefig("docs/review_length_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Length by Sentiment

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for label, color, name in [(0, "#e74c3c", "Negative"), (1, "#2ecc71", "Positive")]:
    subset = train_df[train_df["label"] == label]["word_count"].clip(upper=500)
    subset.plot.kde(ax=ax, color=color, label=name, lw=2)
ax.set_xlabel("Word Count")
ax.set_title("Review Length Density by Sentiment")
ax.legend()
plt.tight_layout()
plt.show()

## Top Words by Sentiment

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer as SklearnCV

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, (label, title, color) in zip(axes, [(0, "Negative Reviews", "#e74c3c"), (1, "Positive Reviews", "#2ecc71")]):
    texts = train_df[train_df["label"] == label]["combined_text"].sample(50000, random_state=42)
    cv = SklearnCV(max_features=30, stop_words="english")
    X = cv.fit_transform(texts)
    word_counts = X.sum(axis=0).A1
    words = cv.get_feature_names_out()
    sorted_idx = word_counts.argsort()
    ax.barh(words[sorted_idx], word_counts[sorted_idx], color=color)
    ax.set_title(f"Top 30 Words \u2014 {title}")
plt.tight_layout()
plt.savefig("docs/top_words.png", dpi=150, bbox_inches="tight")
plt.show()

## Word Clouds

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (label, title, cmap) in zip(axes, [(0, "Negative Reviews", "Reds"), (1, "Positive Reviews", "Greens")]):
    text = " ".join(train_df[train_df["label"] == label]["combined_text"].sample(50000, random_state=42))
    wc = WordCloud(width=800, height=400, background_color="white", colormap=cmap, max_words=200).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(title, fontsize=14)
    ax.axis("off")
plt.tight_layout()
plt.savefig("docs/wordclouds.png", dpi=150, bbox_inches="tight")
plt.show()

## Title vs Text Length

In [ ]:
train_df["title_len"] = train_df["title"].str.len()
train_df["text_len"] = train_df["text"].str.len()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(train_df["title_len"].clip(upper=200), bins=50, alpha=0.6, label="Title", color="steelblue")
ax.hist(train_df["text_len"].clip(upper=3000), bins=50, alpha=0.6, label="Text", color="coral")
ax.set_xlabel("Character Length")
ax.set_ylabel("Count")
ax.set_title("Title vs Text Length Distribution")
ax.legend()
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
summary = train_df[["char_length", "word_count", "title_len", "text_len"]].describe().round(1)
print(summary.to_string())